# Actividad 1 — El principio del tamaño y el juego de los números
### IIC3800 · Modelos de aprendizaje en NCC e IA · clase del 31 de agosto

**15 minutos.** Las tareas 1 y 2 son para la clase; la 3 queda **para la casa**.
El código de abajo ya funciona: es el modelo del *number game*
(juego de los números) de Tenenbaum y Griffiths. Ejecuta la primera celda y pasa
directamente a las tres tareas.

Recuerda las dos ecuaciones:

$$P(X \mid h)=\left(\frac{1}{|h|}\right)^{n} \quad\text{(muestreo fuerte / strong sampling)}
\qquad
P(y \in C \mid X)=\sum_{h \ni y} P(h \mid X) \quad\text{(promedio de hipótesis)}$$

In [ ]:
import numpy as np, matplotlib.pyplot as plt

N = 100
U = np.arange(1, N+1)

def hipotesis_matematicas():
    """34 hipótesis: 5 + 10 múltiplos + 9 potencias + 10 terminaciones."""
    H = {"impares": set(U[U % 2 == 1]), "pares": set(U[U % 2 == 0]),
         "cuadrados": {k*k for k in range(1, 11)},
         "cubos": {k**3 for k in range(1, 5)},
         "primos": {n for n in U if n > 1 and all(n % k for k in range(2, int(n**.5)+1))}}
    for k in range(3, 13):
        H[f"múltiplos de {k}"] = set(U[U % k == 0])
    for k in range(2, 11):
        s = {k**j for j in range(1, 8) if k**j <= N}
        if len(s) > 1: H[f"potencias de {k}"] = s
    for d in range(10):
        H[f"terminan en {d}"] = {n for n in U if n % 10 == d}
    return H

def hipotesis_intervalos():
    """5050 intervalos {a,...,b}."""
    return {f"[{a},{b}]": set(range(a, b+1)) for a in U for b in range(a, N+1)}

MAT, INT = hipotesis_matematicas(), hipotesis_intervalos()
print(f"{len(MAT)} hipótesis matemáticas + {len(INT)} intervalos")

def prior(lam=0.5, erlang=True, sigma=10):
    """lam de la masa a la clase matemática (uniforme dentro),
    1-lam a los intervalos con densidad Erlang sobre |h|."""
    P = {k: lam/len(MAT) for k in MAT}
    w = {k: ((len(s)/sigma**2)*np.exp(-len(s)/sigma) if erlang else 1.0)
         for k, s in INT.items()}
    Z = sum(w.values())
    for k in INT: P[k] = (1-lam)*w[k]/Z
    return P

def posterior(X, P, fuerte=True):
    post = {}
    for k, s in list(MAT.items()) + list(INT.items()):
        if not set(X) <= s: continue
        post[k] = P[k] * ((1/len(s))**len(X) if fuerte else 1.0)
    Z = sum(post.values())
    return {k: v/Z for k, v in post.items()}

def generaliza(X, P, fuerte=True):
    """P(y en C | X) para todo y — ecuación del promedio de hipótesis."""
    post = posterior(X, P, fuerte)
    todas = dict(MAT); todas.update(INT)
    g = np.zeros(N+1)
    for k, pk in post.items():
        for y in todas[k]: g[y] += pk
    return g, post

def mostrar(X, P=None, fuerte=True, top=5):
    P = prior() if P is None else P
    g, post = generaliza(X, P, fuerte)
    plt.figure(figsize=(9, 2))
    cols = ["crimson" if k in X else "navy" for k in range(1, 101)]
    plt.bar(range(1, 101), g[1:101], color=cols, width=0.85)
    plt.ylim(0, 1.05); plt.xlabel("y"); plt.ylabel("P(y ∈ C | X)")
    plt.title(f"X = {X}   ({'fuerte' if fuerte else 'débil'})"); plt.show()
    for k, v in sorted(post.items(), key=lambda z: -z[1])[:top]:
        print(f"   {k:22s} {v:.3f}")

P = prior()
mostrar([16], P)

---
## Tarea 1

Encuentra **dos** conjuntos de a lo más cuatro números que produzcan comportamientos
cualitativamente distintos: uno tipo **regla** (*rule-like*, todo o nada) y otro tipo
**similitud** (*similarity*, gradiente). Después enuncia en una frase qué propiedad de
los ejemplos decide cuál de los dos aparece.

*(No basta con copiar los del paper. Encuentra otros y explica el criterio.)*

In [ ]:
# tu código aquí


---
## Tarea 2

Cambia el muestreo a **débil** (*weak sampling*, `fuerte=False`): la verosimilitud pasa a
valer 1 para toda hipótesis consistente. Corre los tres conjuntos clásicos
`[16]`, `[16,8,2,64]`, `[16,23,19,20]`.

¿Cuál de los tres patrones sobrevive y cuál se destruye? Explícalo con la razón de
verosimilitudes $(|h_2|/|h_1|)^n$ — no con palabras generales.

In [ ]:
# tu código aquí


---
## Tarea 3 · Para la casa

El espacio de hipótesis matemáticas está **escrito a mano**: alguien decidió que
"potencias de 2" está y "números cuyos dígitos suman 7" no está.

Encuentra un conjunto de ejemplos para el cual el modelo prediga algo que tú
rechazarías como humano. Luego di exactamente qué le falta (o qué le sobra) al espacio
$\mathcal{H}$ para arreglarlo, y si ese arreglo se puede justificar sin mirar los datos
del experimento.

Esta es la objeción central a todo el marco, y es el tema del resto de la clase.
No hace falta terminarla en clase: tráela pensada.

In [ ]:
# tu código aquí
